# **05_colombia_cleaning**

## GOAL
- The data that we have in Neon is raw, so here we're going to clean it and load it into the processed schema in Neon.
- The idea is to clean the data to answer this question: What is the state of credit delinquency in Colombia today, how has it changed since 2015, and in which types of credit and institutions is it concentrated?
- We're going to bring the data, look at it, clean it, check it and save it in processed.
- At the end, we'll have a clean table with one row per entity, month and product, with the calculated delinquency rate.
- The next step is to move the cleaning process to src/ so it runs in GitHub Actions.

## Part 1 – Bring the raw data

In [1]:
import sys
sys.path.append("..")
import pandas as pd
from src.load import get_engine

In [2]:
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)

In [3]:
engine = get_engine()

In [4]:
df_raw = pd.read_sql("SELECT * FROM raw.co_loan_portfolio", engine) # Bring the raw data from Neon

In [5]:
df_raw.shape # verify how many rows and columns the dataset has

(112602, 34)

In [6]:
df_raw.dtypes # Check the data type of each column

tipo_entidad                   str
codigo_entidad                 str
nombreentidad                  str
fecha_corte                    str
unicap                         str
descrip_uc                     str
renglon                        str
desc_renglon                   str
_1_saldo_de_la_cartera_a       str
_2_vigente                     str
_3_vencida_1_2_meses           str
_4_vencida_2_3_meses           str
_5_vencida_1_3_meses           str
_6_vencida_3_4_meses           str
_7_vencida_de_4_meses          str
_8_vencida_3_6_meses           str
_9_vencida_6_meses             str
_10_vencida_1_4_meses          str
_11_vencida_4_6_meses          str
_12_vencida_6_12_meses         str
_13_vencida_12_18_meses        str
_14_vencida_12_meses           str
_15_vencida_18_meses           str
_16_n_mero_de_clientes_mora    str
_17_calificaci_n_de_riesgo     str
_18_calificaci_n_de_riesgo     str
_19_calificaci_n_de_riesgo     str
_20_calificaci_n_de_riesgo     str
_21_calificaci_n_de_

## Part 2 – Look at the data

In [7]:
df_raw.head(3)

,tipo_entidad,codigo_entidad,nombreentidad,fecha_corte,unicap,descrip_uc,renglon,desc_renglon,_1_saldo_de_la_cartera_a,_2_vigente,_3_vencida_1_2_meses,_4_vencida_2_3_meses,_5_vencida_1_3_meses,_6_vencida_3_4_meses,_7_vencida_de_4_meses,_8_vencida_3_6_meses,_9_vencida_6_meses,_10_vencida_1_4_meses,_11_vencida_4_6_meses,_12_vencida_6_12_meses,_13_vencida_12_18_meses,_14_vencida_12_meses,_15_vencida_18_meses,_16_n_mero_de_clientes_mora,_17_calificaci_n_de_riesgo,_18_calificaci_n_de_riesgo,_19_calificaci_n_de_riesgo,_20_calificaci_n_de_riesgo,_21_calificaci_n_de_riesgo,_22_calificaci_n_de_riesgo,_23_calificaci_n_de_riesgo,_24_calificaci_n_de_riesgo,_25_calificaci_n_de_riesgo,_26_calificaci_n_de_riesgo
0,1,12,Banco Gnb Sudameris,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,37651351298,34981440061,768061743,579918756,0,0,0,1077855545,244075193,0,0,0,0,0,0,469,6176,32878868825.55,270,1835206170,142,975343138,326,1368695340,69,593093538
1,1,6,Itaú Banco Itaú.,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,476766070943.32,453506953108.61,8052110662.22,3950013841.33,0,0,0,6973931783.54,4283061547.62,0,0,0,0,0,0,2476,43553,442277573569.18,1356,13043155083.47,951,7559380077.26,965,8135341560.27,671,5750620653.14
2,1,9,Citibank,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,826852228376.59,782016598063.64,9271463390.82,8065650991.89,0,0,0,17024347599.27,10474168330.97,0,0,0,0,0,0,5830,125128,772666404396.07,1840,11740703688.96,1486,9872854504.87,1854,14111818486.48,2469,18460447300.21


In [8]:
print(df_raw.isna().sum().sum())    # Check for any NULL value or empty value
print((df_raw == "").sum().sum())

0
0


In [9]:
df_raw["descrip_uc"].nunique()

32

In [10]:
df_raw["descrip_uc"].value_counts()

descrip_uc
TARJETAS DE CRÉDITO                        11137
CARTERA COMERCIAL LEASING FINANCIERO        9296
LIBRANZA                                    9052
CARTERA COMERCIAL CONSTRUCCION              8007
CARTERA COMERCIAL MICROEMPRESA              5271
LIBRE INVERSIÓN                             5133
CARTERA COMERCIAL PYMES                     5029
CARTERA COMERCIAL EMPRESARIAL               4867
CARTERA COMERCIAL CORPORATIVO               4580
CREDITOS DE CONSUMO PARA EMPLEADOS          4089
VEHÍCULO                                    3449
VIVIENDA NO VIS PESOS                       3406
OTROS PORTAFOLIOS DE CONSUMO                3380
MICROCREDITOS < O IGUALES A 25 SMMLV        3183
VIVIENDA VIS PESOS                          2962
MICROCREDITOS > 25  Y HASTA 120 SMMLV       2946
CREDITOS DE VIVIENDA PARA EMPL NO VIS       2909
CARTERA COMERCIAL FINAN E INSTITUCIO        2751
CRÉDITO ROTATIVO                            2732
CARTERA COMERCIAL OFICIAL O GOBIERNO        2515
CREDITOS 

In [11]:
df_raw["desc_renglon"].nunique()

57

In [12]:
df_raw["desc_renglon"].value_counts()

desc_renglon
MICROEMPRESA                                5271
LIBRE INVERSIÓN                             5133
PYMES                                       5029
EMPRESARIAL                                 4867
CORPORATIVO                                 4580
LIBRANZA TOTAL                              4243
LIBRANZA OTROS                              4242
VEHÍCULO                                    3449
TARJETA DE CRÉDITO TOTAL                    3430
NO VIS PESOS TOTAL                          3406
CONSUMO OTROS TOTAL                         3380
TARJETA DE CRÉDITO INGRESO HASTA 2 SMMLV    3276
CRÉDITOS DE CONSUMO PARA EMPLEADOS          3275
MICRO MENOR TOTAL                           3183
VIS PESOS TOTAL                             2962
MICRO MAYOR TOTAL                           2946
FINANCIERO E INSTITUCIONAL                  2751
CRÉDITO ROTATIVO TOTAL                      2732
TARJETA CREDITO INGRESO MAYOR 2SMMLV        2655
CRÉDITO EMPLEADOS VIVIENDA NO VIS           2622
CONSTRU

### FINDINGS

- The dataset has no NULL values or empty values
- We've found that `descrip_uc` is a group, and inside each group there are one or more products described in `desc_renglon`
- There are 32 groups and 57 products.
- All columns are stored as text, so we need to convert the numbers and the date.
- `fecha_corte` includes a time (`00:00:00`) that we don’t need. We’ll keep only the date.
- 19 products are subtotals (they end in TOTAL). Some groups only have the TOTAL row, so we can’t delete all of them. Rule: keep one row per entity, month and group: the TOTAL row if the group has one, or the only row if it doesn’t.
- 32 groups are too many for a chart, so we’ll put them into 4 credit types: commercial, consumer, housing and microcredit.
- Product names in `desc_renglon` have typos and abbreviations (for example `PROYRCTOS`, `LEASIN`), so we’ll use the group names in `descrip_uc` instead.
- Column names like `_1_saldo_de_la_cartera_a` are hard to read. We’ll rename them in English.